In [3]:
import polars as pl
import os

In [4]:
import os
import polars as pl

BASE_DIR   = "../../data"
RAW_DIR    = os.path.join(BASE_DIR, "BASt", "RawData")
OUTPUT_DIR = os.path.join(BASE_DIR, "BASt", "AggregatedDataForSH")
YEARS      = [2021, 2022, 2023, 2024, 2025]
VARIANTS   = ["A", "B"] # A = Autobahn, B = Bundesstraße


# cuts Meta- and Raw-Data files down to SH rows only
# the naming of the original files from BASt.de must not be changed for the function to work!
# after taking a look into one Meta-Data file it can be assumed that "SH" has the "Landesnummer" = 1
def SH_Aggregation():
    for year in YEARS:
        for street_type in VARIANTS:
            source = os.path.join(RAW_DIR,    f"Data {year}", f"Roh_{year}_{street_type}_S")
            target = os.path.join(OUTPUT_DIR, f"Data {year}", f"SH_{year}_{street_type}_S")

            for month in range(1, 13):
                month_str = f"{month:02d}"

                # dynamically creating the path to a source-file
                data_path    = os.path.join(source, f"Roh_{year}_{month_str}_{street_type}_S.csv")
                meta_path    = os.path.join(source, f"Meta_{year}_{month_str}_{street_type}_S.csv")

                # dynamically creating the path where to save the new .csv-file
                SH_data_path = os.path.join(target, f"Roh_{year}_{month_str}_{street_type}_S.csv")
                SH_meta_path = os.path.join(target, f"Meta_{year}_{month_str}_{street_type}_S.csv")

                # checking the Meta-Data of each month if the assumption holds, that 1 is the "Landesnummer" for "SH"
                meta_df = pl.read_csv(meta_path, encoding="latin1", separator=";")
                non_correct_meta = (
                    meta_df
                    .filter((pl.col("Landesnummer") == 1) & (pl.col("Landeskuerzel") != "SH"))
                    .height
                )

                if non_correct_meta == 0:
                    print(f"File Meta_{year}_{month_str}_{street_type}_S.csv is giving '1' as the ID for 'SH'.")

                    # cut Meta-Data files to rows for Schleswig-Holstein only and save as new .csv
                    meta_df.filter(
                        (pl.col("Landesnummer") == 1) & (pl.col("Landeskuerzel") == "SH")
                    ).write_csv(SH_meta_path)
                    print(f"File Meta_{year}_{month_str}_{street_type}_S.csv got processed and aggregated to only SH rows.")

                    # cut Raw-Data files to rows for Schleswig-Holstein only and save as new .csv
                    pl.scan_csv(data_path, separator=";").filter(
                        pl.col("Land") == 1
                    ).sink_csv(SH_data_path)
                    print(f"File Roh_{year}_{month_str}_{street_type}_S.csv got processed and aggregated to only SH rows.")

                else:
                    print(f"----->Month {month_str} could not be aggregated for SH because the meta data is different as expected!")

    print("--- All processes done ---")


if __name__ == "__main__":
    SH_Aggregation()

FileNotFoundError: [Errno 2] No such file or directory: '..\\..\\data\\BASt\\RawData\\Data 2021\\Roh_2021_A_S\\Meta_2021_01_A_S.csv'